# 01 — DuckDB Columnar Storage Intuition

**Goal:** Understand why columnar storage (DuckDB + Parquet) is faster than row storage (SQLite) for analytical queries.

When a query only touches a few columns out of many, a columnar engine like DuckDB reads just
those column stripes from Parquet. A row-oriented engine like SQLite must scan every column of
every row, even if the query only needs two.

This notebook builds a synthetic dataset mirroring our Marshall Fire zonal-statistics table,
loads it into both DuckDB (Parquet) and SQLite, and benchmarks the same analytical query on each.

In [5]:
import sqlite3
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

In [6]:
# ---------------------------------------------------------------------------
# Create a synthetic zonal_stats table
#   4,800 parcels x 5 observation dates = 24,000 rows
#   Columns mirror what we will compute in later notebooks.
# ---------------------------------------------------------------------------

np.random.seed(42)

N_PARCELS = 4800
DATES = ["2021-11-15", "2022-01-15", "2022-06-15", "2023-06-15", "2024-06-15"]

parcel_ids = np.repeat(np.arange(1, N_PARCELS + 1), len(DATES))
obs_dates = np.tile(DATES, N_PARCELS)

df = pd.DataFrame({
    "parcel_id": parcel_ids,
    "observation_date": obs_dates,
    "vv_change_db": np.random.normal(-2.0, 3.0, len(parcel_ids)),
    "vh_change_db": np.random.normal(-1.5, 2.8, len(parcel_ids)),
    "dnbr": np.random.normal(0.15, 0.25, len(parcel_ids)),
    "swir_b7": np.random.uniform(0.05, 0.45, len(parcel_ids)),
    "ndvi": np.random.normal(0.35, 0.2, len(parcel_ids)),
    "parcel_area_m2": np.random.uniform(200, 5000, len(parcel_ids)),
    "officially_destroyed": np.random.choice([0, 1], size=len(parcel_ids), p=[0.7, 0.3]),
})

out_dir = Path("../data/interim")
out_dir.mkdir(parents=True, exist_ok=True)

# -- Write Parquet (columnar) -----------------------------------------------
parquet_path = out_dir / "zonal_stats_synthetic.parquet"
df.to_parquet(parquet_path, index=False)

# -- Write SQLite (row-oriented) --------------------------------------------
sqlite_path = out_dir / "zonal_stats_synthetic.sqlite"
sqlite_path.unlink(missing_ok=True)
sqlite_conn = sqlite3.connect(str(sqlite_path))
df.to_sql("zonal_stats", sqlite_conn, index=False, if_exists="replace")
sqlite_conn.execute("CREATE INDEX idx_obs_date ON zonal_stats(observation_date)")
sqlite_conn.commit()

print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
print(f"Parquet size: {parquet_path.stat().st_size / 1024:.1f} KB")
print(f"SQLite size:  {sqlite_path.stat().st_size / 1024:.1f} KB")
df.head()

Task was destroyed but it is pending!
task: <Task pending name='Task-158' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/fhsu/Documents/projects/marshall_fire/.venv/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-159' coro=<Kernel.shell_main() running at /Users/fhsu/Documents/projects/marshall_fire/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/fhsu/Documents/projects/marshall_fire/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/fhsu/Documents/projects/marshall_fire/.venv/lib/python3.11/site-packages/IPython/core/compilerop.py:86: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)
Task was destroyed but it is pending!
task: <Task pending name='Task-159' coro=<Kernel.shell_main() running at /Users/fhsu/Document

Rows: 24,000  |  Columns: 9
Parquet size: 1444.2 KB
SQLite size:  2224.0 KB


,parcel_id,observation_date,vv_change_db,vh_change_db,dnbr,swir_b7,ndvi,parcel_area_m2,officially_destroyed
0,1,2021-11-15,-0.509858,-3.379846,0.301893,0.294019,0.182089,4334.978810,0
1,1,2022-01-15,-2.414793,-3.498639,-0.032212,0.078097,0.150789,3773.387605,1
2,1,2022-06-15,-0.056934,2.491137,0.082637,0.146402,0.613227,4900.841365,1
3,1,2023-06-15,2.569090,3.720358,0.095038,0.164346,0.425975,3763.425988,0
4,1,2024-06-15,-2.702460,-4.100871,0.146140,0.195173,0.299578,479.937119,0


## Compare DuckDB (columnar) vs SQLite (row-oriented)

We run the same analytical query — touching only 2 of the 9 columns — against
both engines. DuckDB reads from Parquet (columnar), SQLite from its row-oriented
B-tree storage. The query groups by `observation_date` and computes the average
`vv_change_db`.

**Why DuckDB wins here:** Parquet stores each column in a separate stripe.
DuckDB reads only the `observation_date` and `vv_change_db` stripes, skipping
the other 7 columns entirely. SQLite must read full rows from disk even though
it only needs 2 fields — every row drags along all 9 columns.

In [7]:
ANALYTICAL_QUERY = """
SELECT observation_date,
       AVG(vv_change_db) AS mean_vv_change,
       COUNT(*)          AS n
FROM   {table}
GROUP  BY observation_date
ORDER  BY observation_date
"""

duck_con = duckdb.connect()
N_ITER = 50

# -- DuckDB + Parquet (columnar) -------------------------------------------
duck_sql = ANALYTICAL_QUERY.format(table=f"'{parquet_path}'")
t0 = time.perf_counter()
for _ in range(N_ITER):
    result_duck = duck_con.execute(duck_sql).fetchdf()
elapsed_duck = (time.perf_counter() - t0) / N_ITER * 1000

# -- SQLite (row-oriented) -------------------------------------------------
sqlite_sql = ANALYTICAL_QUERY.format(table="zonal_stats")
cursor = sqlite_conn.cursor()
t0 = time.perf_counter()
for _ in range(N_ITER):
    cursor.execute(sqlite_sql)
    result_sqlite = cursor.fetchall()
elapsed_sqlite = (time.perf_counter() - t0) / N_ITER * 1000

print(f"DuckDB + Parquet avg: {elapsed_duck:.2f} ms")
print(f"SQLite (row-based) avg: {elapsed_sqlite:.2f} ms")
print(f"Ratio (SQLite / DuckDB): {elapsed_sqlite / elapsed_duck:.1f}x")
print()
print(result_duck)

DuckDB + Parquet avg: 1.32 ms
SQLite (row-based) avg: 2.41 ms
Ratio (SQLite / DuckDB): 1.8x

  observation_date  mean_vv_change     n
0       2021-11-15       -1.960681  4800
1       2022-01-15       -1.935114  4800
2       2022-06-15       -2.049804  4800
3       2023-06-15       -2.044231  4800
4       2024-06-15       -1.982108  4800


## Scale up to see the gap widen

At 24,000 rows both engines are fast because everything fits in cache.
Let's scale to 500,000 rows — the columnar advantage becomes more pronounced
as the amount of irrelevant data that SQLite must drag through grows.

In [8]:
# ---------------------------------------------------------------------------
# Build a 500K-row version and repeat the benchmark
# ---------------------------------------------------------------------------

N_BIG = 500_000
np.random.seed(99)

df_big = pd.DataFrame({
    "parcel_id": np.arange(N_BIG),
    "observation_date": np.random.choice(DATES, N_BIG),
    "vv_change_db": np.random.normal(-2.0, 3.0, N_BIG),
    "vh_change_db": np.random.normal(-1.5, 2.8, N_BIG),
    "dnbr": np.random.normal(0.15, 0.25, N_BIG),
    "swir_b7": np.random.uniform(0.05, 0.45, N_BIG),
    "ndvi": np.random.normal(0.35, 0.2, N_BIG),
    "parcel_area_m2": np.random.uniform(200, 5000, N_BIG),
    "officially_destroyed": np.random.choice([0, 1], N_BIG, p=[0.7, 0.3]),
})

# -- Parquet ----------------------------------------------------------------
big_pq = out_dir / "zonal_stats_big.parquet"
df_big.to_parquet(big_pq, index=False)

# -- SQLite -----------------------------------------------------------------
big_sqlite = out_dir / "zonal_stats_big.sqlite"
big_sqlite.unlink(missing_ok=True)
big_sqlite_conn = sqlite3.connect(str(big_sqlite))
df_big.to_sql("zonal_stats", big_sqlite_conn, index=False, if_exists="replace")
big_sqlite_conn.execute("CREATE INDEX idx_obs_date ON zonal_stats(observation_date)")
big_sqlite_conn.commit()

print(f"Big Parquet: {big_pq.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Big SQLite:  {big_sqlite.stat().st_size / 1024 / 1024:.1f} MB")

# -- Benchmark -------------------------------------------------------------
N_ITER = 20

# DuckDB + Parquet
duck_big_sql = ANALYTICAL_QUERY.format(table=f"'{big_pq}'")
t0 = time.perf_counter()
for _ in range(N_ITER):
    duck_con.execute(duck_big_sql).fetchdf()
elapsed_big_duck = (time.perf_counter() - t0) / N_ITER * 1000

# SQLite
big_cursor = big_sqlite_conn.cursor()
sqlite_big_sql = ANALYTICAL_QUERY.format(table="zonal_stats")
t0 = time.perf_counter()
for _ in range(N_ITER):
    big_cursor.execute(sqlite_big_sql)
    big_cursor.fetchall()
elapsed_big_sqlite = (time.perf_counter() - t0) / N_ITER * 1000

print(f"\n500K rows -- DuckDB + Parquet: {elapsed_big_duck:.2f} ms")
print(f"500K rows -- SQLite:           {elapsed_big_sqlite:.2f} ms")
print(f"Ratio (SQLite / DuckDB):       {elapsed_big_sqlite / elapsed_big_duck:.1f}x")

big_sqlite_conn.close()

Big Parquet: 26.9 MB
Big SQLite:  46.0 MB

500K rows -- DuckDB + Parquet: 2.66 ms
500K rows -- SQLite:           68.86 ms
Ratio (SQLite / DuckDB):       25.9x


## Verify dbt can read Parquet

dbt-duckdb uses DuckDB under the hood and can query Parquet files as external
sources. Here we confirm that our file is well-formed by reading it back and
inspecting the schema.

In [9]:
# ---------------------------------------------------------------------------
# Read the parquet back through DuckDB and inspect the schema
# ---------------------------------------------------------------------------

schema = duck_con.execute(f"DESCRIBE SELECT * FROM '{parquet_path}'").fetchdf()
print("Schema of zonal_stats_synthetic.parquet:\n")
print(schema.to_string(index=False))

row_count = duck_con.execute(f"SELECT COUNT(*) FROM '{parquet_path}'").fetchone()[0]
print(f"\nRow count: {row_count:,}")

# Quick sanity query -- same style a dbt model would use
sample = duck_con.execute(f"""
    SELECT observation_date,
           ROUND(AVG(vv_change_db), 3) AS mean_vv,
           ROUND(AVG(ndvi), 3)         AS mean_ndvi,
           SUM(officially_destroyed)    AS n_destroyed
    FROM   '{parquet_path}'
    GROUP  BY observation_date
    ORDER  BY observation_date
""").fetchdf()
print("\nSample aggregation:\n")
print(sample.to_string(index=False))

duck_con.close()
sqlite_conn.close()
print("\nDone — Parquet file is valid and queryable.")

Schema of zonal_stats_synthetic.parquet:

         column_name column_type null  key default extra
           parcel_id      BIGINT  YES None    None  None
    observation_date     VARCHAR  YES None    None  None
        vv_change_db      DOUBLE  YES None    None  None
        vh_change_db      DOUBLE  YES None    None  None
                dnbr      DOUBLE  YES None    None  None
             swir_b7      DOUBLE  YES None    None  None
                ndvi      DOUBLE  YES None    None  None
      parcel_area_m2      DOUBLE  YES None    None  None
officially_destroyed      BIGINT  YES None    None  None

Row count: 24,000

Sample aggregation:

observation_date  mean_vv  mean_ndvi  n_destroyed
      2021-11-15   -1.961      0.349       1406.0
      2022-01-15   -1.935      0.352       1465.0
      2022-06-15   -2.050      0.347       1403.0
      2023-06-15   -2.044      0.353       1454.0
      2024-06-15   -1.982      0.351       1463.0

Done — Parquet file is valid and queryable.
